# SwinIR (jetsr_swin) — Google Colab GPU Training

Transformer-based super-resolution (SwinIR, optional FiLM conditioning) for CMS jet images, on the **same parquet data** as the multiscale-SR GAN — so results are directly comparable.

## Before you Run-All
- **(P1)** Push the `swinir/` folder to GitHub; set `REPO_BRANCH` in Cell 2.
- **(P2)** Put the `*.parquet` jet files where Colab can read them (Google Drive easiest); set `DATA_DIR` in Cell 4.
- **(P3)** *Runtime -> Change runtime type -> GPU* (T4/L4/A100 all fine).
- **(P4)** Optional W&B: add `WANDB_API_KEY` (and optionally `WANDB_PROJECT`) via the Secrets panel or Cell 5.

## How to run
1. Set `DATA_DIR` (Cell 4).
2. Pick `CONFIG` (base vs FiLM) and `EPOCHS` (Cell 6).
3. *Runtime -> Run all*.

## Cell 1 — Environment check (GPU is required)

In [ ]:
import subprocess, sys
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit(
        "No CUDA GPU. Enable it: Runtime -> Change runtime type -> Hardware accelerator = GPU, "
        "then Runtime -> Run all again."
    )
print("device:", torch.cuda.get_device_name(0))
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

## Cell 2 — Get the code (clone repo at a pinned branch)
Re-running is safe: the clone dir is removed first.

In [ ]:
import os, shutil, subprocess
from pathlib import Path

REPO_URL    = "https://github.com/rajveer43/cms-superres-reconstruction.git"
REPO_BRANCH = "master"                 # <- match the branch you pushed swinir to
CLONE_DIR   = Path("/content/repo")
CODE_DIR    = CLONE_DIR / "swinir"     # SwinIR project root

if CLONE_DIR.exists():
    shutil.rmtree(CLONE_DIR)

r = subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(CLONE_DIR)],
    capture_output=True, text=True,
)
print(r.stdout, r.stderr)
if r.returncode != 0:
    raise SystemExit(f"git clone failed (branch '{REPO_BRANCH}'). Check the branch/repo.")
if not CODE_DIR.exists():
    raise SystemExit(f"Expected code at {CODE_DIR} but it is missing after clone.")

os.chdir(CODE_DIR)
_commit = subprocess.run(["git", "-C", str(CLONE_DIR), "rev-parse", "HEAD"],
                         capture_output=True, text=True).stdout.strip()
print("cwd    :", os.getcwd())
print("commit :", _commit, "(branch", REPO_BRANCH + ")")

## Cell 3 — Dependencies
Colab ships torch/numpy/pyarrow/matplotlib/pyyaml. We add `wandb` + `python-dotenv`. Torch untouched.

In [ ]:
import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    try:
        importlib.import_module(import_name or pkg)
        print(f"ok: {pkg}")
    except ImportError:
        print(f"installing: {pkg}")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

for pkg, imp in [("wandb", "wandb"), ("python-dotenv", "dotenv")]:
    ensure(pkg, imp)

import torch, numpy, pyarrow
print("torch", torch.__version__, "| numpy", numpy.__version__, "| pyarrow", pyarrow.__version__)

## Cell 4 — Locate the parquet data (Google Drive)
Same jet parquet files as the multiscale-SR run (columns `X_jets_LR` / `X_jets`). **Edit `DATA_DIR`.**

In [ ]:
from pathlib import Path
import pyarrow.parquet as pq

from google.colab import drive
drive.mount("/content/drive")

# ================== EDIT THIS ONE LINE ==================
DATA_DIR = Path("/content/drive/MyDrive/datasets")   # <- folder that CONTAINS the *.parquet files
# =======================================================

files = sorted(DATA_DIR.glob("*.parquet"))
if not files:
    raise SystemExit(f"No *.parquet in {DATA_DIR}. Fix DATA_DIR (list with: !ls \"{DATA_DIR}\").")
total = 0
print("DATA_DIR:", DATA_DIR)
for f in files:
    n = pq.ParquetFile(f).metadata.num_rows
    total += n
    print(f"  {f.name}: {n:,} rows")
print(f"TOTAL: {total:,} jets across {len(files)} file(s)")

## Cell 5 — W&B (optional)
SwinIR reads the project from `$WANDB_PROJECT` (falls back to `jetsr-swin`). Set `WANDB_API_KEY` (and optionally `WANDB_PROJECT`) via the Colab Secrets panel, or paste below. Use the **same** `WANDB_PROJECT` you used locally so runs land together for comparison.

In [ ]:
import os

USE_WANDB = False
try:
    from google.colab import userdata
    _key = userdata.get("WANDB_API_KEY")
    if _key:
        os.environ["WANDB_API_KEY"] = _key
        USE_WANDB = True
        print("W&B: key found in Colab Secrets -> logging ENABLED")
    try:
        _proj = userdata.get("WANDB_PROJECT")
        if _proj:
            os.environ["WANDB_PROJECT"] = _proj
    except Exception:
        pass
except Exception as e:
    print("W&B: no Colab secret ->", type(e).__name__)

if not USE_WANDB:
    _PASTED_KEY = ""       # <- optionally paste your wandb key
    _PASTED_PROJECT = ""   # <- optionally set your wandb project (else uses the config/default)
    if _PASTED_KEY:
        os.environ["WANDB_API_KEY"] = _PASTED_KEY
        USE_WANDB = True
        if _PASTED_PROJECT:
            os.environ["WANDB_PROJECT"] = _PASTED_PROJECT
        print("W&B: using pasted key -> logging ENABLED")

print("WANDB_PROJECT =", os.environ.get("WANDB_PROJECT", "(unset -> default jetsr-swin)"))
if not USE_WANDB:
    print("W&B: DISABLED (training will pass --no-wandb).")

## Cell 6 — Parameters (edit these)
Pick the config: `swinir_base.yaml` (plain SwinIR, apples-to-apples vs the GAN) or `swinir_film.yaml` (with FiLM class conditioning). `data_dir` and `output_dir` are overridden below so the config's relative paths don't matter on Colab.

In [ ]:
CONFIG    = "configs/swinir_base.yaml"     # or "configs/swinir_film.yaml"
EPOCHS    = 20
OUTPUT_DIR = "/content/outputs/swinir_run"

import yaml, tempfile
from pathlib import Path

# Load the chosen config, override data_dir/epochs/output_dir, write a temp config.
_cfg = yaml.safe_load(Path(CONFIG).read_text())
_cfg["data_dir"]   = str(DATA_DIR)
_cfg["epochs"]     = EPOCHS
_cfg["output_dir"] = OUTPUT_DIR
if not USE_WANDB:
    _cfg["wandb_enabled"] = False
CONFIG_RESOLVED = str(Path(tempfile.gettempdir()) / "swinir_colab_config.yaml")
Path(CONFIG_RESOLVED).write_text(yaml.safe_dump(_cfg))
print("resolved config ->", CONFIG_RESOLVED)
print("  data_dir  :", _cfg["data_dir"])
print("  epochs    :", _cfg["epochs"])
print("  output_dir:", _cfg["output_dir"])
print("  model     :", "FiLM" if _cfg.get("model", {}).get("use_film") else "base SwinIR")

## Cell 7 — Train (streamed logs)
Runs `scripts/train_swinir.py` on the resolved config. AMP is on (fp16) for CUDA. Watch `val_l1`, `val_psnr`, and `energy_response` — the same metrics as the GAN, so you can compare the two models directly.

In [ ]:
import subprocess, sys, os
from pathlib import Path

cmd = [sys.executable, "scripts/train_swinir.py", "--config", CONFIG_RESOLVED]
if not USE_WANDB:
    cmd += ["--no-wandb"]
print("RUN:", " ".join(cmd), "\n", flush=True)

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise SystemExit(f"training failed (exit {proc.returncode}) — see the log above.")

OUT = Path(OUTPUT_DIR)
print("\nOUTPUT_DIR:", OUT)
ckpts = sorted((OUT / "checkpoints").glob("*.pt"))
print("checkpoints:", [c.name for c in ckpts])

## Cell 8 — Show figures, copy to Drive, zip for download

In [ ]:
import shutil
from pathlib import Path
from IPython.display import Image, display

OUT = Path(OUTPUT_DIR)
# Display any sample/metric PNGs the trainer produced.
for png in sorted(OUT.rglob("*.png"))[:6]:
    print(png.relative_to(OUT))
    display(Image(filename=str(png)))

# 1) Copy the run to Drive so it persists after the session ends.
try:
    drive_out = Path("/content/drive/MyDrive/swinir_runs") / OUT.name
    if drive_out.exists():
        shutil.rmtree(drive_out)
    shutil.copytree(OUT, drive_out)
    print("\nCopied run to Drive:", drive_out)
except Exception as e:
    print("\n(could not copy to Drive:", type(e).__name__, e, ")")

# 2) Zip and offer a direct download.
zip_base = str(Path("/content") / OUT.name)
shutil.make_archive(zip_base, "zip", root_dir=str(OUT))
print("Archive:", zip_base + ".zip")
try:
    from google.colab import files
    files.download(zip_base + ".zip")
except Exception as e:
    print("(auto-download unavailable:", type(e).__name__, "— grab it from Files panel or Drive)")

## Cell 9 — Comparing SwinIR vs the multiscale-SR GAN

Both models train on the same parquet data and log the same core metrics (`val_l1`, `val_psnr`, `energy_response`), so compare them directly:

- **In W&B:** point both runs at the same entity and compare the metric panels. (SwinIR project = `$WANDB_PROJECT`; the GAN project = `multiscale-sr`.)
- **Note the difference in setup:** SwinIR here is a fixed 64->125 (or config `out_size`) SR, no GAN/adversarial term — it's the reconstruction-only transformer baseline. The multiscale GAN additionally has adversarial + energy-weighted losses across 16/32/64x. Keep that in mind when reading PSNR/L1 side by side.

**Resume / another config:** set `RESUME` is not wired here; to try FiLM, set `CONFIG = "configs/swinir_film.yaml"` in Cell 6 and *Run all* again. For disconnect-proof long runs, set `OUTPUT_DIR` to a Drive path so checkpoints land on Drive live.